# 3D-ARM-Gaze: data quality and trial selection

This notebook inspects the canonical collection produced by `scripts/prepare_arm_gaze.py`. Each observation is one contiguous target presentation, identified by participant, experimental condition, and original target number. These trials include reaching and target holding; they are not velocity-segmented movement primitives.

The release has 20 right-handed participants and three conditions: initial acquisition, return to neutral posture (RNP) after pauses, and RNP after target pairs. The default collection follows the authors' **custom (displayed teleoperated) arm** convention. The corrected sensor-derived virtual arm is separately available using `--arm virtual`; the two should not be pooled as equivalent measurements.

No SOC run, learned gain, or other notebook output is needed. Preprocessing must first finish and write `manifest.json`.

This notebook audits the input cohort. Phase 1 representation and endpoint-dependence analysis live in `02_3D_ARM_Gaze_Preprocessing.ipynb`. Both notebooks independently read the canonical collection.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from motion_primitives.paths import PROJECT_ROOT
from motion_primitives.arm_gaze import PHASES

collection = PROJECT_ROOT / 'collections/3D-ARM-Gaze/custom-phase200-v1'
output = PROJECT_ROOT / 'notebooks/results/00_3D_ARM_Gaze_Quality' / collection.name
output.mkdir(parents=True, exist_ok=True)
manifest = json.loads((collection / 'manifest.json').read_text())
metadata = pd.read_csv(collection / 'movements.csv')
audit = pd.read_csv(collection / 'trial_audit.csv').fillna({'exclusion_reason': 'retained'})
recordings = pd.read_csv(collection / 'recordings.csv')
print(f'{len(metadata):,} trials; {metadata.subject.nunique()} participants; {len(manifest['joint_names'])} angles in radians')
print(manifest['coordinates'])

17,539 trials; 20 participants; 7 angles in radians
DBAS22 custom arm; reflected Unity x; relative XZY/X/YXZ angles


## Selection and observed quality

The cohort excludes the authors' rejected targets, pauses plus two following samples, neutral-posture intervals, and incomplete final targets. An interruption within a target rejects that trial rather than joining disjoint pieces. Unsuccessful trials and both members of repeated target pairs remain; `validated` records the success counter increment.

Sample-level tracking annotations are preserved as flags. They are not an additional exclusion criterion: applying every sample flag to a whole trial would produce a different cohort from the authors' target-level policy. Gaze is valid only when its eye validity equals 31. Invalid eye vectors and invalid binocular focus are NaN in the saved context arrays. The published eye-validity flags were reconstructed after acquisition, not recorded directly.

Inspect counts, trial durations, gaze availability, and tracking flags before selecting a scientific cohort. Differences here describe retained observations; they do not establish a causal effect of the condition.

In [2]:
summary = metadata.groupby('action').agg(
    trials=('movement_id', 'size'), subjects=('subject', 'nunique'),
    validated_fraction=('validated', 'mean'), median_duration_s=('duration_seconds', 'median'),
    min_duration_s=('duration_seconds', 'min'), max_duration_s=('duration_seconds', 'max'),
    left_gaze_valid_fraction=('left_gaze_valid_fraction', 'mean'),
    right_gaze_valid_fraction=('right_gaze_valid_fraction', 'mean'),
    author_flagged_samples=('author_flagged_samples', 'sum')).reindex(PHASES)
display(summary)
display(pd.crosstab(audit.action, audit.exclusion_reason).reindex(PHASES))
summary.to_csv(output / 'condition_summary.csv')
print('Gaze fractions above are trial-weighted. Original timing is retained in native.npz.')
display(metadata.loc[metadata.angle_wrap_events > 0, ['movement_id', 'angle_wrap_events']])

Gaze fractions above are trial-weighted. Original timing is retained in native.npz.


                              trials  ...  author_flagged_samples
action                                ...                        
initial_acquisition             5809  ...                   17832
test_RNP_after_pauses           3913  ...                    9759
test_RNP_after_targets_pairs    7817  ...                   12957

[3 rows x 9 columns]

exclusion_reason              author_excluded_target  ...  retained
action                                                ...          
initial_acquisition                              291  ...      5809
test_RNP_after_pauses                             75  ...      3913
test_RNP_after_targets_pairs                      90  ...      7817

[3 rows x 5 columns]

                                            movement_id  angle_wrap_events
8162     3D-ARM-Gaze/s10/initial_acquisition/target-140                  1
8163     3D-ARM-Gaze/s10/initial_acquisition/target-141                  1
8353   3D-ARM-Gaze/s10/test_RNP_after_pauses/target-049                  1
8415   3D-ARM-Gaze/s10/test_RNP_after_pauses/target-157                  1
8416   3D-ARM-Gaze/s10/test_RNP_after_pauses/target-158                  1
8546  3D-ARM-Gaze/s10/test_RNP_after_targets_pairs/t...                  1
8743  3D-ARM-Gaze/s10/test_RNP_after_targets_pairs/t...                  1

## Timing, tracking availability, and a native trajectory

The figure shows retained trial counts and success labels, physical durations, and the fraction of valid samples for each eye. Gaze availability is averaged within each trial and then across trials, so every trial has equal weight. A median-duration trial shows all seven arm angles against its original elapsed timestamps.

Seven trials needed a principal-angle branch correction during dataset preparation. Their identities are listed above. Sample-level tracking flags are retained separately; a discontinuity still requires inspection of those flags and the original rotations. This notebook measures data quality without adding exclusions or changing the canonical collection.

In [3]:
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})
labels = ['Initial acquisition', 'RNP after pauses', 'RNP after target pairs']
colors = ['#0072B2', '#D55E00', '#009E73']
fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
x = np.arange(3)
successes = metadata.groupby('action').validated.sum().reindex(PHASES)
axes[0, 0].bar(x, successes, color=colors, label='Validated')
axes[0, 0].bar(x, summary.trials - successes, bottom=successes, color=colors,
               alpha=.3, hatch='//', label='Not validated')
axes[0, 0].set(xticks=x, xticklabels=labels, ylabel='Retained target trials', title='Cohort and success labels')
axes[0, 0].legend()
for task, label, color in zip(PHASES, labels, colors):
    axes[0, 1].hist(metadata.loc[metadata.action == task, 'duration_seconds'], bins=np.linspace(0, 6, 61),
                    histtype='step', linewidth=2, density=True, color=color, label=label)
axes[0, 1].set(xlabel='Native trial duration (s)', ylabel='Probability density (1/s)', title='Reaching plus target holding')
axes[0, 1].legend()
axes[1, 0].bar(x - .18, summary.left_gaze_valid_fraction, .36, label='Left eye', color='#0072B2')
axes[1, 0].bar(x + .18, summary.right_gaze_valid_fraction, .36, label='Right eye', color='#E69F00')
axes[1, 0].set(xticks=x, xticklabels=labels, ylim=(0, 1), ylabel='Mean within-trial valid fraction',
               title='Eye-tracking availability (trial-weighted)')
axes[1, 0].legend()
index = (metadata.duration_seconds - metadata.duration_seconds.median()).abs().idxmin()
with np.load(collection / 'native.npz') as native:
    assert native['movement_ids'][index] == metadata.loc[index, 'movement_id']
    start, stop = native['offsets'][index:index + 2]
    time = native['time'][start:stop].copy()
    angles = native['q'][start:stop].copy()
for channel, name in enumerate(manifest['joint_names']):
    axes[1, 1].plot(time, angles[:, channel], label=name.replace('_', ' '))
axes[1, 1].set(xlabel='Elapsed trial time (s)', ylabel='Joint angle (rad)', title=metadata.loc[index, 'movement_id'])
axes[1, 1].legend(fontsize=9, ncol=2)
for ax in axes.flat:
    ax.grid(axis='y', alpha=.2)
fig.savefig(output / 'dataset_overview.png', dpi=180)
fig.savefig(output / 'dataset_overview.svg')
plt.show()

<Figure size 1400x900 with 4 Axes>

## Interpretation and source access

The conditions contain different target protocols, and the paired condition intentionally repeats poses around neutral-posture resets. Trial counts are not counts of independent participants. Subject-held-out evaluation should split on `subject`, and condition comparisons should account for paired targets rather than treating them as unrelated actions.

World positions use Unity's left-handed frame, whose origin/orientation is arbitrary across participants. The exported joint angles use the authors' reflected-frame relative rotations; neither the joint labels nor their coordinate conventions are interchangeable with ReachGrasp's Vicon angles. Do not concatenate those joint spaces without an explicit anatomical mapping.

`context/<subject>/<phase>.npz` contains native target/head/end-effector/gaze channels. Use each metadata row's `source_start_index:source_stop_index` to align it with the movement. Directions are unit vectors; positions are in metres; quaternions use xyzw order. No gaze interpolation or fabricated finger kinematics is included.

Definitions are supplied in `../3D-ARM-Gaze/DBAS22_DocOnline/MainDataExplained.pdf` and `SummaryOfFiles.pdf`; the seven-angle convention is implemented in the supplied `rot_quat_utils.py:quats2config`.